# Train Cogsworth on Kaggle

Fine-tunes a small open model (Qwen2.5 0.5B Instruct by default) with LoRA on Cogsworth's grounded training data,
scores it against the engine, then exports browser-ready weights for transformers.js and scores those too.

**Before you run:**
1. **Settings → Accelerator → GPU T4** (one T4 is enough; with T4 x2 the notebook uses one).
2. **Settings → Internet → On** (needs a phone-verified Kaggle account). The code, the base model and the
   packages download from the internet. The notebook clones `REPO_REF` of the repo, so it always trains on the
   current `portfolio-data.js`. With internet off it falls back to an attached dataset holding the repo or the kit.
3. **Save Version → Save & Run All.** With the 0.5B model the run takes about an hour and a half: ~25 minutes
   of training, the rest answering, exporting and scoring.

Results land in `/kaggle/working`: `cogsworth-out.zip` (adapter, answers, browser model) and `report.md`
(every eval answer side by side). The score tables print in the log.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4; must be set before torch loads
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONUNBUFFERED"] = "1"

BASE = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-1.5B-Instruct": better answers, ~3x slower to train, ~1.5 GB in the browser
EPOCHS = 3
EXPORT_FORMATS = "q8"                 # q8 (~0.5 GB for 0.5B) is the one to ship; add ",q4" to also build q4 (larger for Qwen)
REPO_REF = "fix-kaggle-oom"                    # branch, tag or commit of the repo to train from
for k, v in {"BASE": BASE, "EPOCHS": str(EPOCHS), "EXPORT_FORMATS": EXPORT_FORMATS}.items():
    os.environ[k] = v   # the shell cells below read these

In [ ]:
# Get the code: clone the repo at REPO_REF; without internet, use an attached dataset instead.
import glob, shutil, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/repo")
WORK = Path("/kaggle/working/cogsworth-train")
for p in (REPO, WORK):
    shutil.rmtree(p, ignore_errors=True)
try:
    subprocess.run(["git", "clone", "-q", "https://github.com/PujaSridhar/PujaSridhar.github.io.git", str(REPO)], check=True, timeout=300)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_REF], check=True)
    print("Cloned the repo at", REPO_REF, subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
except Exception as error:
    print("Could not clone the repo (", error, "), looking for an attached dataset instead")
    shutil.rmtree(REPO, ignore_errors=True)
    hits = [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/data/train.jsonl", recursive=True)]
    hits = [h for h in hits if (h / "python" / "train_lora.py").exists()]
    if not hits:
        raise SystemExit("No internet and no dataset with the kit attached")
    print("Using dataset:", hits[0])
    if (hits[0].parent.parent / "portfolio-data.js").exists():
        shutil.copytree(hits[0].parent.parent, REPO, ignore=shutil.ignore_patterns("node_modules", ".git"))
    else:
        REPO = None   # only the kit: training works, scoring is skipped
        shutil.copytree(hits[0], WORK, ignore=shutil.ignore_patterns("__pycache__", "out"))
if REPO:
    shutil.copytree(REPO / "cogsworth" / "train", WORK, ignore=shutil.ignore_patterns("__pycache__", "out", "node_modules"))
os.environ["REPO_SRC"] = str(REPO or "")
for f in ["train.jsonl", "val.jsonl", "eval-prompts.jsonl"]:
    print(f, sum(1 for _ in open(WORK / "data" / f)), "rows")

In [ ]:
%%bash
# Packages. torch comes with Kaggle's image (CUDA build), so it is not reinstalled.
# Kaggle's preinstalled diffusers needs a newer huggingface_hub than transformers 4.x allows, and optimum's
# ONNX exporter imports it if present, so it is removed (nothing here uses it).
# pip may print dependency warnings about gradio; they don't affect this notebook.
set -euo pipefail
pip uninstall -y -q diffusers || true
pip install -q -r /kaggle/working/cogsworth-train/python/requirements.txt
nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Fine-tune (LoRA, fp16 on the T4)

Only the assistant replies are trained on; the system prompt, retrieved context and question are masked.
Watch `eval_loss` after each epoch: it should drop and then flatten. The best epoch is kept.

In [ ]:
%%bash
# Each shell cell stops the run on the first error (set -e), so a failed step can't be skipped silently.
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
python train_lora.py --base "$BASE" --data ../data --out /kaggle/working/out --epochs "$EPOCHS" \
  --precision fp16 --batch 1 --grad-accum 16

## 2. Answer the eval questions

Answers from the trained model and from the untuned base model, so you can see what training changed.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
python generate.py --base "$BASE" --adapter /kaggle/working/out/adapter --out /kaggle/working/out/trained-answers.jsonl --label "trained model"
python generate.py --base "$BASE" --out /kaggle/working/out/base-answers.jsonl --label "base model"

In [ ]:
# A quick look at a few answers
import json
rows = [json.loads(l) for l in open("/kaggle/working/out/trained-answers.jsonl")]
base = {(r["suite"], r["index"]): r["answer"] for r in map(json.loads, open("/kaggle/working/out/base-answers.jsonl"))}
for r in rows[:12]:
    print("Q:", r["q"])
    print("  trained:", r["answer"])
    print("  base:   ", base.get((r["suite"], r["index"]), "")[:300])
    print()

## 3. Score the answers

`score.js` grades every answer against the eval suites, next to the engine. Runs on CPU in Node.

In [ ]:
%%bash
set -euo pipefail
if [ -z "$REPO_SRC" ]; then echo "Only the kit is available, not the repo, so scoring is skipped. Score on your laptop (README)."; exit 0; fi
NODE=v22.12.0
if ! command -v node >/dev/null || [ "$(node -p 'process.versions.node.split(".")[0]')" -lt 22 ]; then
  curl -fsSL https://nodejs.org/dist/$NODE/node-$NODE-linux-x64.tar.xz | tar -xJ -C /opt
  ln -sf /opt/node-$NODE-linux-x64/bin/node /opt/node-$NODE-linux-x64/bin/npm /usr/local/bin/
fi
node --version
cd "$REPO_SRC"
npm --prefix cogsworth install --no-audit --no-fund --loglevel=error
node cogsworth/train/score.js --base /kaggle/working/out/base-answers.jsonl --trained /kaggle/working/out/trained-answers.jsonl
cp cogsworth/train/eval/out/report.md /kaggle/working/report.md

In [ ]:
%%bash
# Package what exists so far, so a problem in the export below can't lose it.
set -euo pipefail
cd /kaggle/working/out
zip -qr /kaggle/working/cogsworth-out.zip adapter trained-answers.jsonl base-answers.jsonl training-summary.json
ls -lh /kaggle/working/cogsworth-out.zip

## 4. Export for the browser

Merges the adapter into the base model, exports ONNX with optimum and quantizes to q8. The folder
`cogsworth-web` loads directly with transformers.js. Capped at 40 minutes so a stuck export can't burn GPU quota.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/cogsworth-train/python
timeout 40m python -u export_onnx.py --base "$BASE" --adapter /kaggle/working/out/adapter --out /kaggle/working/out/cogsworth-web --formats "$EXPORT_FORMATS"
cd /kaggle/working/out
rm -rf checkpoints merged onnx-fp32
zip -qr /kaggle/working/cogsworth-out.zip cogsworth-web
ls -lh /kaggle/working/cogsworth-out.zip

## 5. Score the browser model

The exported q8 model as the site would run it, through transformers.js, with and without the grounding guard
that falls back to the engine. Capped at 60 minutes.

In [ ]:
%%bash
set -euo pipefail
if [ -z "$REPO_SRC" ]; then echo "Scoring skipped (no repo)."; exit 0; fi
cd "$REPO_SRC"
timeout 60m node cogsworth/train/score.js --base /kaggle/working/out/base-answers.jsonl --trained /kaggle/working/out/trained-answers.jsonl \
  --onnx /kaggle/working/out/cogsworth-web --dtype q8
cp cogsworth/train/eval/out/report.md /kaggle/working/report.md
echo "Download cogsworth-out.zip and report.md from the Output tab, then follow 'After training' in cogsworth/train/README.md"